In [1]:
import random
from datasets import load_dataset, get_dataset_config_names
import numpy as np
from typing import Iterable

# =============================================================================
# 💡 데이터셋 개요: Genshin Voice
# 📝 의미: 인기 게임 '원신(Genshin Impact)'의 캐릭터 음성 대사(Voice Lines)를 수집한 데이터셋입니다.
# ✨ 용도: 음성 인식(ASR), 음성 분류(Audio Classification), 텍스트 음성 변환(TTS) 등 음성 AI 연구에 사용됩니다.
# 🎯 목표: 우리는 이 데이터가 가진 여러 가지 '맥락(Context)' 정보를 파악하고, 오디오와 텍스트가 어떻게 연결되어 있는지 탐색하는 실습을 해보겠습니다.
# =============================================================================

DATASET_NAME = "simon3000/genshin-voice"
SAMPLE_COUNT = 10 # 초보자 실습을 위해 10개의 샘플만 사용합니다.

# 1. 데이터셋 Config 로드 (필수 절차)
# =============================================================================
print("========================================================")
print("💖 Genshin Voice 데이터셋 탐험을 시작합니다! 💖")
print("========================================================")

try:
    configs = get_dataset_config_names(DATASET_NAME)
    print(f"✅ 사용 가능한 Config 목록: {configs}")
    
    # 기본 설정 사용
    selected_config = configs[0]
    print(f"✨ 기본 설정 '{selected_config}'을 사용합니다.")

except Exception as e:
    print(f"ℹ️ Config 로드 중 오류 발생: {e}. 기본 설정을 사용합니다.")
    selected_config = None

# 2. 데이터 로딩 전략: 스트리밍 (Streaming) 우선 시도
# =============================================================================
dataset = None
try:
    print("\n➡️ [STEP 1/4] 데이터 로딩을 시도합니다... (Streaming 모드)")
    # 🚀 메모리 효율성을 위해 스트리밍 모드로 전체 데이터셋을 로드합니다.
    dataset = load_dataset(DATASET_NAME, split='train', streaming=True)
    print("🎉 성공! 스트리밍 모드로 데이터셋을 로드했습니다. (대용량 데이터 처리 가능)")

except Exception as e:
    # 🛑 스트리밍 모드에서 문제가 생길 경우, 일반 모드로 전환하여 적은 양만 다운로드합니다.
    print(f"🚨 스트리밍 로드 실패 ({e}). 일반 모드로 전환하여 소량만 다운로드합니다.")
    try:
        dataset = load_dataset(DATASET_NAME, split='train', streaming=False)
    except Exception as e_fallback:
        print(f"🛑 데이터 로드에 실패했습니다. 필수 환경 라이브러리를 확인해주세요. ({e_fallback})")
        exit()

# 3. 샘플링 및 반복자(Iterator) 준비 (핵심 로직)
# =============================================================================

# ⭐ [필수] 데이터셋 크기 확인이나 인덱스 슬라이싱(dataset[:10])은 스트리밍 모드에서 불가능합니다.
# ⭐ [필수] 반드시 .take()로 샘플을 지정한 후, iter()로 반복자를 만들어야 합니다.

if hasattr(dataset, "take"):
    print(f"\n⭐ [샘플링] 전체 데이터셋 중 상위 {SAMPLE_COUNT}개 샘플을 추출합니다.")
    # take()는 iterable을 반환하므로, 반복자로 변환해야 함.
    sampled_dataset_iterator: Iterable = iter(dataset.take(SAMPLE_COUNT))
else:
    print("⚠️ 데이터셋이 반복자가 아닙니다. 스크립트를 종료합니다.")
    exit()

# 4. 데이터 분석 및 실습 시뮬레이션 (메인 로직)
# =============================================================================

print("\n========================================================")
print("✨ [STEP 2/4] 데이터셋의 메타데이터 분석: 어떤 캐릭터가 어떤 대사를 했는지 볼까요?")
print("========================================================")

# 데이터셋에서 모든 'speaker' 값과 'language' 값을 추출하여 고유 값만 분석합니다.
unique_speakers = set()
unique_languages = set()
unique_types = set()

sample_data_list = []
sample_count = 0

print("🔍 현재 샘플을 순회하며 주요 속성(Speaker, Language, Type)을 수집 중...")

# iter()를 사용하여 반복적으로 샘플을 추출
while sample_count < SAMPLE_COUNT:
    try:
        # next() 함수를 사용해 한 개씩 데이터를 가져옵니다. (가장 안전한 방식)
        sample = next(sampled_dataset_iterator)
        sample_data_list.append(sample)
        
        # 🎯 분석: Speaker와 Language 추출
        unique_speakers.add(sample.get('speaker', 'N/A'))
        unique_languages.add(sample.get('language', 'N/A'))
        unique_types.add(sample.get('type', 'N/A'))

        sample_count += 1

    except StopIteration:
        print("😢 모든 샘플 처리가 완료되었습니다.")
        break
    except Exception as e:
        print(f"🚨 데이터 처리 중 예외 발생: {e}")
        break

print("\n--- 📊 데이터셋 통계 분석 결과 (전체 데이터의 특징) ---")
print(f"🗣️ 등장하는 주요 캐릭터(Speaker) 개수: {len(unique_speakers)}명")
print(f"🌐 지원하는 언어(Language) 종류: {list(unique_languages)}")
print(f"📜 대사 유형(Type) 종류: {list(unique_types)}")
print("---------------------------------------------------\n")


print("========================================================")
print("🎙️ [STEP 3/4] 핵심 실습: ASR/TTS 맥락 분석 (Focus on the Pair)")
print("========================================================")

print("👩‍💻 AI 관점: 이 데이터셋은 오디오(Audio)와 텍스트(Transcription)가 짝을 이루는 '쌍(Pair)'으로 구성되어 있습니다.")
print("   우리가 하려는 작업은 '이 오디오가 어떤 텍스트를 의미하는가'를 파악하는 것입니다.")

# 가장 첫 번째 샘플을 꺼내와 구조를 파악합니다.
if sample_data_list:
    sample_0 = sample_data_list[0]
    
    print(f"\n✨ [첫 번째 샘플] 분석 시작...")
    print(f"  - 캐릭터: {sample_0.get('speaker', 'N/A')} | 유형: {sample_0.get('type', 'N/A')}")
    print(f"  - 언어: {sample_0.get('language', 'N/A')} | 원본 파일명: {sample_0.get('inGameFilename', 'N/A')}")

    print("\n--- 🧠 ASR 시뮬레이션 관점 ---")
    
    # 1. 오디오 데이터의 존재 확인 (Audio 객체)
    audio_data = sample_0.get('audio')
    if audio_data:
        # 오디오 데이터의 형식 정보를 출력합니다.
        print(f"  ✅ 오디오 데이터 확인: {type(audio_data).__name__}")
        # 실제 값(wav 파일)을 분석하는 것은 로컬 환경이 필요하므로, 메타데이터만 사용합니다.
        print("  [결과] 오디오 데이터 객체: 실제 음성 파일(WAV)을 담고 있으며, 시간적 정보를 가지고 있습니다.")
    else:
        print("  ❌ 오디오 데이터가 없습니다.")

    # 2. 텍스트 데이터 분석 (Transcription)
    transcription = sample_0.get('transcription', 'N/A')
    if transcription != 'N/A':
        print(f"\n  ✅ 트랜스크립션(Transcription, 텍스트) 분석:")
        print(f"     - 내용: '{transcription[:40]}...'")
        print("     - 의미: 이 텍스트가 오디오가 어떤 음성인지를 알려주는 '정답지' 역할을 합니다.")
    else:
        print("\n  ⚠️ 트랜스크립션(Transcription) 텍스트 정보가 없습니다.")
        
    # 3. 전체 맥락 파악
    print("\n⭐ 최종 결론: 성공적인 AI 처리를 위해 필요한 정보")
    print("   👉 오디오 데이터(음성) + 트랜스크립션(텍스트) + 언어/캐릭터 정보(맥락)가 모두 갖춰져야 합니다.")

# 5. 데이터 탐색 및 LLM 활용 예시 (Creative Use Case)
# =============================================================================
print("\n========================================================")
print("🔮 [STEP 4/4] 창의적 응용: 캐릭터별 대화 패턴 분석")
print("========================================================")

# 데이터 분석을 통해 'Speaker'와 'Type'의 관계를 탐색합니다.
speaker_type_pairings = {}
for sample in sample_data_list:
    speaker = sample.get('speaker')
    dialogue_type = sample.get('type')
    
    if speaker and dialogue_type:
        if speaker not in speaker_type_pairings:
            speaker_type_pairings[speaker] = set()
        speaker_type_pairings[speaker].add(dialogue_type)

print("✨ 데이터셋 분석을 통해 각 캐릭터(Speaker)가 어떤 종류의 대사(Type)를 주로 하는지 패턴을 발견할 수 있습니다.")

# LLM에 입력할 가상의 프롬프트 생성 예시
print("\n--- 📝 LLM (Large Language Model) 프롬프트 생성 시뮬레이션 ---")
print("   (데이터셋의 정보를 LLM이 이해할 수 있도록 구조화하는 과정입니다.)")

# 무작위로 하나의 캐릭터와 타입 조합을 선택
random_speaker = random.choice(list(speaker_type_pairings.keys()))
random_types = list(speaker_type_pairings[random_speaker])
random_type = random.choice(random_types)

print(f"\n[분석 대상]: 캐릭터 '{random_speaker}'의 '{random_type}' 유형 대사.")
print("---------------------------------------------------------")

prompt_template = f"""
[CONTEXT]: character={random_speaker}, type={random_type}, language={list(unique_languages)[0]}
[AUDIO]: (여기에 WAV 오디오 데이터가 삽입됩니다.)
[QUESTION]: 이 캐릭터가 이 유형의 대사를 할 때 일반적으로 사용하는 감정적 톤(emotion tone)은 무엇인가요? (예: 활기참, 냉정함, 슬픔)
"""
print("💡 생성된 LLM 프롬프트 구조:")
print(prompt_template.strip())
print("\n✅ 이처럼 데이터셋의 모든 메타 정보(캐릭터, 상황, 언어)를 조합하여 AI에게 풍부한 '맥락(Context)'을 제공할 수 있습니다.")

print("\n\n========================================================")
print("🏆 실습 완료! Genshin Voice 데이터셋 탐험을 성공적으로 마치셨습니다!")
print("   데이터 구조화와 메타데이터 분석 능력을 키운 훌륭한 실습이었습니다!")
print("========================================================")

/home/kipi009/.local/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


💖 Genshin Voice 데이터셋 탐험을 시작합니다! 💖
✅ 사용 가능한 Config 목록: ['default']
✨ 기본 설정 'default'을 사용합니다.

➡️ [STEP 1/4] 데이터 로딩을 시도합니다... (Streaming 모드)
🎉 성공! 스트리밍 모드로 데이터셋을 로드했습니다. (대용량 데이터 처리 가능)

⭐ [샘플링] 전체 데이터셋 중 상위 10개 샘플을 추출합니다.

✨ [STEP 2/4] 데이터셋의 메타데이터 분석: 어떤 캐릭터가 어떤 대사를 했는지 볼까요?
🔍 현재 샘플을 순회하며 주요 속성(Speaker, Language, Type)을 수집 중...
🚨 데이터 처리 중 예외 발생: No module named 'numba'

--- 📊 데이터셋 통계 분석 결과 (전체 데이터의 특징) ---
🗣️ 등장하는 주요 캐릭터(Speaker) 개수: 0명
🌐 지원하는 언어(Language) 종류: []
📜 대사 유형(Type) 종류: []
---------------------------------------------------

🎙️ [STEP 3/4] 핵심 실습: ASR/TTS 맥락 분석 (Focus on the Pair)
👩‍💻 AI 관점: 이 데이터셋은 오디오(Audio)와 텍스트(Transcription)가 짝을 이루는 '쌍(Pair)'으로 구성되어 있습니다.
   우리가 하려는 작업은 '이 오디오가 어떤 텍스트를 의미하는가'를 파악하는 것입니다.

🔮 [STEP 4/4] 창의적 응용: 캐릭터별 대화 패턴 분석
✨ 데이터셋 분석을 통해 각 캐릭터(Speaker)가 어떤 종류의 대사(Type)를 주로 하는지 패턴을 발견할 수 있습니다.

--- 📝 LLM (Large Language Model) 프롬프트 생성 시뮬레이션 ---
   (데이터셋의 정보를 LLM이 이해할 수 있도록 구조화하는 과정입니다.)


IndexError: Cannot choose from an empty sequence